# FineWeb Parquet Reader & Explorer

This notebook provides a **memory-efficient** way to explore and clean FineWeb parquet data.

**Key design decisions:**
- **Lazy loading**: reads one row group (~1000 rows) at a time, never loads the full 2GB file
- **Multi-file support**: auto-detects all `.parquet` files in the data directory
- **Cleaning pipeline**: modular, configurable filters applied per row group
- **Scalable**: same code works for 5 files or 500 files

**Data**: FineWeb (HuggingFace) — cleaned Common Crawl web text, ~173 row groups per slice, 9 columns.

## 1. Setup & Data Discovery

In [ ]:
import os
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np
from collections import Counter
import hashlib
import re
import warnings
warnings.filterwarnings('ignore')

# --- Auto-detect data directory ---
# Works on both JupyterHub (/data/fineweb) and local machines
_candidates = [
    Path('/data/fineweb'),
    Path('data/fineweb'),
    Path('.'),
]

DATA_DIR = None
for d in _candidates:
    try:
        if d.exists() and list(d.glob('*.parquet')):
            DATA_DIR = d
            break
    except Exception:
        pass

if DATA_DIR is None:
    raise FileNotFoundError(
        'No parquet files found. Check data directory.\n'
        f'Searched: {[str(d) for d in _candidates]}'
    )

FILES = sorted(DATA_DIR.glob('*.parquet'))
print(f'Data directory : {DATA_DIR.resolve()}')
print(f'Parquet files  : {len(FILES)}')
print()
total_size = 0
for f in FILES:
    size_mb = f.stat().st_size / (1024 ** 2)
    total_size += size_mb
    pf = pq.ParquetFile(f)
    est_rows = sum(pf.metadata.row_group(j).num_rows for j in range(pf.num_row_groups))
    print(f'  [{FILES.index(f):>2}] {f.name:30s}  {size_mb:>8,.1f} MB  |  {pf.num_row_groups:>3} groups  |  ~{est_rows:>8,} rows')
print(f'\nTotal: {total_size:,.1f} MB across {len(FILES)} file(s)')

## 2. Schema Exploration

All slices share the same schema. Let's inspect it once.

In [ ]:
pf = pq.ParquetFile(FILES[0])

print('Columns and types:')
print('-' * 50)
for i, field in enumerate(pf.schema_arrow):
    print(f'  {i}. {field.name:20s} {field.type}')

print(f'\nRow group 0 has {pf.metadata.row_group(0).num_rows} rows')
print(f'Total row groups: {pf.num_row_groups}')

## 3. Lazy Sampling & Preview

**Why lazy?** Each parquet file is ~2GB with ~173 row groups.
Loading the whole file needs 4-6GB RAM (arrow + pandas copy).
Instead, we read individual row groups (~1000 rows each) on demand.

This is the core pattern we'll use throughout.

In [ ]:
def read_group(file_idx=0, group_idx=0):
    """Read a single row group as a pandas DataFrame."""
    pf = pq.ParquetFile(FILES[file_idx])
    return pf.read_row_group(group_idx).to_pandas()


def sample_rows(file_idx=0, n_groups=3, rows_per_group=5):
    """Sample rows by reading a few evenly-spaced row groups."""
    pf = pq.ParquetFile(FILES[file_idx])
    n = min(n_groups, pf.num_row_groups)
    indices = np.linspace(0, pf.num_row_groups - 1, n, dtype=int)

    frames = []
    for gi in indices:
        df = pf.read_row_group(gi).to_pandas().head(rows_per_group)
        df['_group'] = int(gi)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def preview(file_idx=0, group_idx=0, row_idx=0, text_chars=1500):
    """Preview a single record with full text."""
    df = read_group(file_idx, group_idx)
    if row_idx >= len(df):
        print(f'Row {row_idx} out of range (max {len(df)-1})')
        return

    row = df.iloc[row_idx]
    print(f'File: {FILES[file_idx].name} | Group: {group_idx} | Row: {row_idx}')
    print('=' * 100)
    for col in ['url', 'date', 'language', 'language_score', 'token_count', 'id', 'dump']:
        if col in df.columns:
            print(f'{col}: {row[col]}')
    print('-' * 100)
    text = str(row.get('text', ''))
    print(text[:text_chars] + ('\n... [truncated]' if len(text) > text_chars else ''))


# --- Demo: sample 3 groups, 3 rows each ---
sample = sample_rows(0, n_groups=3, rows_per_group=3)
show_cols = [c for c in ['url', 'language', 'language_score', 'token_count', '_group'] if c in sample.columns]
sample[show_cols]

In [ ]:
# Preview a single record in full
preview(file_idx=0, group_idx=0, row_idx=0)

## 4. Statistical Profiling

We iterate through row groups **one at a time** and aggregate stats.
This uses ~constant memory regardless of file size.

Set `max_groups` to limit scan time during exploration (e.g. 10 groups = ~10k rows).

In [ ]:
def profile_file(file_idx=0, max_groups=None):
    """Compute stats by scanning row groups one at a time (memory efficient)."""
    pf = pq.ParquetFile(FILES[file_idx])
    n_groups = pf.num_row_groups if max_groups is None else min(max_groups, pf.num_row_groups)

    lang_counter = Counter()
    token_counts = []
    lang_scores = []
    text_lengths = []
    url_domains = Counter()
    total_rows = 0
    null_counts = Counter()

    for gi in range(n_groups):
        df = pf.read_row_group(gi).to_pandas()
        total_rows += len(df)

        for col in df.columns:
            null_counts[col] += int(df[col].isna().sum())

        if 'language' in df.columns:
            lang_counter.update(df['language'].dropna().tolist())
        if 'token_count' in df.columns:
            token_counts.extend(df['token_count'].dropna().tolist())
        if 'language_score' in df.columns:
            lang_scores.extend(df['language_score'].dropna().tolist())
        if 'text' in df.columns:
            text_lengths.extend(df['text'].dropna().str.len().tolist())
        if 'url' in df.columns:
            for url in df['url'].dropna():
                try:
                    url_domains[url.split('/')[2]] += 1
                except Exception:
                    pass

        del df  # free memory

    # --- Print report ---
    print(f'File: {FILES[file_idx].name}')
    print(f'Scanned: {n_groups}/{pf.num_row_groups} row groups, {total_rows:,} rows')

    print(f'\n--- Null Values ---')
    for col in pf.schema_arrow.names:
        cnt = null_counts.get(col, 0)
        pct = cnt / total_rows * 100 if total_rows else 0
        print(f'  {col:20s}: {cnt:>6,} ({pct:.2f}%)')

    if lang_counter:
        print(f'\n--- Language Distribution (top 10) ---')
        for lang, cnt in lang_counter.most_common(10):
            print(f'  {lang:5s}: {cnt:>8,} ({cnt/total_rows*100:.1f}%)')

    if token_counts:
        tc = np.array(token_counts)
        print(f'\n--- Token Count ---')
        print(f'  min={tc.min()}  p5={np.percentile(tc,5):.0f}  p25={np.percentile(tc,25):.0f}  median={np.median(tc):.0f}  p75={np.percentile(tc,75):.0f}  p95={np.percentile(tc,95):.0f}  max={tc.max()}')

    if lang_scores:
        ls = np.array(lang_scores)
        print(f'\n--- Language Score ---')
        print(f'  min={ls.min():.4f}  p5={np.percentile(ls,5):.4f}  p25={np.percentile(ls,25):.4f}  median={np.median(ls):.4f}  p75={np.percentile(ls,75):.4f}  p95={np.percentile(ls,95):.4f}  max={ls.max():.4f}')

    if text_lengths:
        tl = np.array(text_lengths)
        print(f'\n--- Text Length (chars) ---')
        print(f'  min={tl.min()}  median={np.median(tl):.0f}  mean={tl.mean():.0f}  max={tl.max()}')

    if url_domains:
        print(f'\n--- Top 10 Domains ---')
        for domain, cnt in url_domains.most_common(10):
            print(f'  {domain:40s}: {cnt:>5,}')

    return {'total_rows': total_rows, 'lang_counter': lang_counter,
            'token_counts': token_counts, 'lang_scores': lang_scores, 'text_lengths': text_lengths}


# Profile first file, first 10 groups (fast scan)
stats = profile_file(0, max_groups=10)

## 5. Data Quality Assessment

Before cleaning, we need to understand what "dirty" looks like.

**Potential issues in web-crawled text:**
- Very short text (< 50 tokens) — often navigation fragments or error pages
- Very long text (> 10k tokens) — possible scraping artifacts
- Low language score — garbled or mixed-language content
- Duplicate URLs — same page crawled multiple times
- Boilerplate-heavy text — cookie banners, privacy policies, etc.
- High character repetition — spam or encoding errors

In [ ]:
def assess_quality(file_idx=0, max_groups=5):
    """Identify quality issues by scanning a few row groups."""
    pf = pq.ParquetFile(FILES[file_idx])
    n_groups = min(max_groups, pf.num_row_groups)

    issues = Counter()
    total = 0
    all_urls = []
    samples = {'short': [], 'low_score': [], 'repetitive': []}

    bp_regex = re.compile(
        r'cookie|privacy policy|terms of service|subscribe to our newsletter|'
        r'click here to|all rights reserved',
        re.IGNORECASE
    )

    for gi in range(n_groups):
        df = pf.read_row_group(gi).to_pandas()
        total += len(df)

        for _, row in df.iterrows():
            text = str(row.get('text', '') or '')
            tc = row.get('token_count', 0) or 0
            ls = row.get('language_score', 1.0) or 1.0
            url = str(row.get('url', ''))
            all_urls.append(url)

            if not text.strip():
                issues['empty_text'] += 1
                continue

            if tc < 50:
                issues['very_short (<50 tok)'] += 1
                if len(samples['short']) < 2:
                    samples['short'].append({'url': url, 'tokens': tc, 'text': text[:150]})

            if tc > 10000:
                issues['very_long (>10k tok)'] += 1

            if ls < 0.65:
                issues['low_lang_score (<0.65)'] += 1
                if len(samples['low_score']) < 2:
                    samples['low_score'].append({'url': url, 'score': f'{ls:.4f}', 'text': text[:150]})

            if len(text) > 100 and len(set(text)) / len(text) < 0.03:
                issues['high_repetition'] += 1
                if len(samples['repetitive']) < 2:
                    samples['repetitive'].append({'url': url, 'text': text[:150]})

            words = text.split()
            if words and len(bp_regex.findall(text)) / len(words) > 0.03:
                issues['boilerplate_heavy'] += 1

        del df

    dup_urls = sum(1 for c in Counter(all_urls).values() if c > 1)

    print(f'Quality Assessment: {FILES[file_idx].name}')
    print(f'Scanned: {n_groups} groups, {total:,} rows')
    print('=' * 70)
    for issue, count in issues.most_common():
        print(f'  {issue:30s}: {count:>5,}  ({count/total*100:.1f}%)')
    print(f'  {"duplicate_urls":30s}: {dup_urls:>5,}')

    for cat, items in samples.items():
        if items:
            print(f'\n  >> Examples [{cat}]:')
            for item in items:
                print(f'     {item}')

    return issues


quality = assess_quality(0, max_groups=5)

## 6. Cleaning Pipeline

**Approach**: define a cleaning config, then apply it row-group-by-row-group.
This keeps memory usage constant even for 2GB files.

**Cleaning steps** (all configurable):
1. Remove empty / null text
2. Filter by `token_count` range (remove too-short and too-long)
3. Filter by `language_score` threshold
4. Filter by minimum text length (chars)
5. Keep only specified languages (optional)
6. Deduplicate by URL

Each step can be toggled on/off via the config dict.

In [ ]:
# --- Default cleaning config (adjust thresholds as you explore) ---
CLEAN_CONFIG = {
    'min_tokens': 50,
    'max_tokens': 50000,
    'min_lang_score': 0.65,
    'min_text_length': 100,   # chars
    'languages': None,        # None = keep all, or e.g. ['en']
    'dedup_url': True,
}


def clean_group(df, config=None):
    """Clean a single row group DataFrame. Returns (cleaned_df, removal_counts)."""
    cfg = config or CLEAN_CONFIG
    removed = Counter()
    n0 = len(df)

    # 1. Empty text
    mask = df['text'].notna() & (df['text'].str.strip().str.len() > 0)
    removed['empty_text'] = int((~mask).sum())
    df = df[mask]

    # 2. Token count
    if 'token_count' in df.columns:
        lo = df['token_count'] >= cfg['min_tokens']
        hi = df['token_count'] <= cfg['max_tokens']
        removed['too_few_tokens'] = int((~lo).sum())
        removed['too_many_tokens'] = int((~hi).sum())
        df = df[lo & hi]

    # 3. Language score
    if 'language_score' in df.columns:
        mask = df['language_score'] >= cfg['min_lang_score']
        removed['low_lang_score'] = int((~mask).sum())
        df = df[mask]

    # 4. Text length
    mask = df['text'].str.len() >= cfg['min_text_length']
    removed['short_text'] = int((~mask).sum())
    df = df[mask]

    # 5. Language filter
    if cfg['languages'] and 'language' in df.columns:
        mask = df['language'].isin(cfg['languages'])
        removed['wrong_language'] = int((~mask).sum())
        df = df[mask]

    # 6. URL dedup
    if cfg['dedup_url'] and 'url' in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset='url', keep='first')
        removed['dup_url'] = before - len(df)

    return df, removed


# --- Demo on one row group ---
demo_df = read_group(0, 0)
cleaned, removed = clean_group(demo_df)
print(f'Row group 0: {len(demo_df)} rows -> {len(cleaned)} rows')
print(f'Removed: {dict({k: v for k, v in removed.items() if v > 0})}')

In [ ]:
def clean_file(file_idx=0, config=None, output_path=None):
    """Clean an entire parquet file, one row group at a time.

    Memory usage stays constant regardless of file size.
    Optionally saves the cleaned result to a new parquet file.
    """
    pf = pq.ParquetFile(FILES[file_idx])
    cfg = config or CLEAN_CONFIG
    total_removed = Counter()
    total_original = 0
    total_kept = 0
    clean_tables = []

    for gi in range(pf.num_row_groups):
        df = pf.read_row_group(gi).to_pandas()
        total_original += len(df)

        cleaned, removed = clean_group(df, cfg)
        total_removed += removed
        total_kept += len(cleaned)

        if len(cleaned) > 0:
            clean_tables.append(pa.Table.from_pandas(cleaned, preserve_index=False))

        del df, cleaned

        if (gi + 1) % 50 == 0:
            print(f'  ... {gi+1}/{pf.num_row_groups} groups processed')

    print(f'\nCleaning complete: {FILES[file_idx].name}')
    print(f'  Original : {total_original:>8,} rows')
    print(f'  Kept     : {total_kept:>8,} rows ({total_kept/total_original*100:.1f}%)')
    print(f'  Removed  : {total_original - total_kept:>8,} rows')
    print(f'\n  Breakdown:')
    for reason, count in total_removed.most_common():
        if count > 0:
            print(f'    {reason:25s}: {count:>6,}')

    if output_path and clean_tables:
        combined = pa.concat_tables(clean_tables)
        pq.write_table(combined, output_path)
        size_mb = Path(output_path).stat().st_size / (1024 ** 2)
        print(f'\n  Saved -> {output_path} ({size_mb:.1f} MB)')

    return clean_tables


# Example: clean the first file and save
# clean_file(0, output_path='cleaned_004_00049.parquet')

## 7. Search & Browse

Utility functions for navigating data interactively — run in separate cells as needed.

In [ ]:
def search_text(keyword, file_idx=0, max_groups=10, max_results=20):
    """Search for keyword in text column, scanning row groups lazily."""
    pf = pq.ParquetFile(FILES[file_idx])
    n = min(max_groups, pf.num_row_groups)
    results = []

    for gi in range(n):
        df = pf.read_row_group(gi).to_pandas()
        mask = df['text'].str.contains(keyword, case=False, na=False)
        hits = df[mask].copy()
        hits['_group'] = gi
        results.append(hits)
        del df

        if sum(len(r) for r in results) >= max_results:
            break

    result = pd.concat(results, ignore_index=True).head(max_results)
    print(f'Found {len(result)} matches (scanned {min(gi+1, n)} groups)')

    if len(result) > 0:
        show = [c for c in ['url', 'language', 'token_count', '_group'] if c in result.columns]
        print(result[show].to_string())

    return result


def search_url(keyword, file_idx=0, max_groups=10, max_results=20):
    """Search for keyword in URL column."""
    pf = pq.ParquetFile(FILES[file_idx])
    n = min(max_groups, pf.num_row_groups)
    results = []

    for gi in range(n):
        df = pf.read_row_group(gi).to_pandas()
        mask = df['url'].str.contains(keyword, case=False, na=False)
        hits = df[mask].copy()
        hits['_group'] = gi
        results.append(hits)
        del df

        if sum(len(r) for r in results) >= max_results:
            break

    result = pd.concat(results, ignore_index=True).head(max_results)
    print(f'Found {len(result)} matches')
    return result


def filter_by(file_idx=0, group_idx=0, lang=None, min_tokens=None, max_tokens=None, min_score=None):
    """Filter a row group by multiple criteria. Returns filtered DataFrame."""
    df = read_group(file_idx, group_idx)
    if lang:
        df = df[df['language'] == lang]
    if min_tokens:
        df = df[df['token_count'] >= min_tokens]
    if max_tokens:
        df = df[df['token_count'] <= max_tokens]
    if min_score:
        df = df[df['language_score'] >= min_score]
    print(f'Filtered: {len(df)} rows')
    return df

In [ ]:
# --- Example usage (uncomment and run) ---

# Search for a keyword in text
# results = search_text('machine learning', file_idx=0)

# Preview a specific result
# preview(file_idx=0, group_idx=5, row_idx=3)

# Filter by language and token count
# filtered = filter_by(0, group_idx=0, lang='en', min_tokens=200)

## 8. Batch Processing (All Files)

When ready, apply the cleaning pipeline across all parquet files.
Each file is processed independently, one row group at a time.

In [ ]:
def clean_all(config=None, output_dir='cleaned'):
    """Clean all parquet files and save to output directory."""
    out = Path(output_dir)
    out.mkdir(exist_ok=True)

    for i, f in enumerate(FILES):
        print(f'\n{"="*80}')
        print(f'Processing [{i+1}/{len(FILES)}]: {f.name}')
        output_path = out / f.name
        clean_file(i, config=config, output_path=str(output_path))

    print(f'\n{"="*80}')
    print(f'All done. Cleaned files saved to: {out.resolve()}')


# Uncomment to run full batch cleaning:
# clean_all()

## 9. Export Results (CSV)

Generate CSV reports from the exploration and cleaning above.
These files go into `outputs/` and are ready to upload to the team repo.

In [ ]:
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

# ====================================================================
# 1. data_overview.csv — one row per parquet file with metadata
# ====================================================================
overview_rows = []
for f in FILES:
    pf = pq.ParquetFile(f)
    est_rows = sum(pf.metadata.row_group(j).num_rows for j in range(pf.num_row_groups))
    overview_rows.append({
        'file': f.name,
        'size_mb': round(f.stat().st_size / (1024**2), 1),
        'row_groups': pf.num_row_groups,
        'est_rows': est_rows,
        'columns': len(pf.schema_arrow),
    })
overview_df = pd.DataFrame(overview_rows)
overview_df.to_csv(OUT_DIR / 'data_overview.csv', index=False)
print('data_overview.csv')
print(overview_df.to_string(index=False))

# ====================================================================
# 2. language_distribution.csv — from profiling stats
# ====================================================================
lang_rows = [{'language': lang, 'count': cnt, 'pct': round(cnt / stats['total_rows'] * 100, 2)}
             for lang, cnt in stats['lang_counter'].most_common()]
lang_df = pd.DataFrame(lang_rows)
lang_df.to_csv(OUT_DIR / 'language_distribution.csv', index=False)
print(f'\nlanguage_distribution.csv  ({len(lang_df)} languages)')
print(lang_df.head(10).to_string(index=False))

# ====================================================================
# 3. token_stats.csv — percentile distribution of token counts
# ====================================================================
tc = np.array(stats['token_counts'])
ls = np.array(stats['lang_scores'])
tl = np.array(stats['text_lengths'])
percentiles = [0, 5, 10, 25, 50, 75, 90, 95, 100]
token_stats = pd.DataFrame({
    'percentile': [f'p{p}' for p in percentiles] + ['mean', 'std'],
    'token_count': [np.percentile(tc, p) for p in percentiles] + [tc.mean(), tc.std()],
    'lang_score':  [np.percentile(ls, p) for p in percentiles] + [ls.mean(), ls.std()],
    'text_length': [np.percentile(tl, p) for p in percentiles] + [tl.mean(), tl.std()],
})
token_stats = token_stats.round(2)
token_stats.to_csv(OUT_DIR / 'token_stats.csv', index=False)
print(f'\ntoken_stats.csv')
print(token_stats.to_string(index=False))

# ====================================================================
# 4. quality_report.csv — issues found during quality assessment
# ====================================================================
quality_rows = [{'issue': issue, 'count': count} for issue, count in quality.most_common()]
quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(OUT_DIR / 'quality_report.csv', index=False)
print(f'\nquality_report.csv')
print(quality_df.to_string(index=False))

# ====================================================================
# 5. cleaning_summary.csv — before/after stats per row group (first 10)
# ====================================================================
pf = pq.ParquetFile(FILES[0])
clean_rows = []
for gi in range(min(10, pf.num_row_groups)):
    df = pf.read_row_group(gi).to_pandas()
    cleaned, removed = clean_group(df)
    clean_rows.append({
        'file': FILES[0].name,
        'row_group': gi,
        'original': len(df),
        'kept': len(cleaned),
        'removed': len(df) - len(cleaned),
        'keep_rate': round(len(cleaned) / len(df) * 100, 1),
        **{k: v for k, v in removed.items() if v > 0},
    })
    del df, cleaned
clean_summary = pd.DataFrame(clean_rows).fillna(0)
clean_summary.to_csv(OUT_DIR / 'cleaning_summary.csv', index=False)
print(f'\ncleaning_summary.csv  (first 10 row groups)')
print(clean_summary.to_string(index=False))

# ====================================================================
# 6. sample_cleaned.csv — 100 cleaned sample rows (text truncated to 500 chars)
# ====================================================================
sample_frames = []
for gi in [0, 50, 100]:
    if gi >= pf.num_row_groups:
        continue
    df = pf.read_row_group(gi).to_pandas()
    cleaned, _ = clean_group(df)
    sample_frames.append(cleaned.head(35))
    del df, cleaned
sample_clean = pd.concat(sample_frames, ignore_index=True).head(100)
sample_clean['text'] = sample_clean['text'].str[:500]  # truncate for CSV readability
sample_clean.to_csv(OUT_DIR / 'sample_cleaned.csv', index=False)
print(f'\nsample_cleaned.csv  ({len(sample_clean)} rows, text truncated to 500 chars)')
print(sample_clean[['url', 'language', 'language_score', 'token_count']].head(5).to_string(index=False))

# ====================================================================
print(f'\n{"="*70}')
print(f'All CSV files saved to: {OUT_DIR.resolve()}')
print(f'Files: {[f.name for f in OUT_DIR.glob("*.csv")]}')

## Next Steps

**After Easter — fine-tuning prep:**
- Tune `CLEAN_CONFIG` thresholds based on quality assessment results
- Add text-level cleaning (strip boilerplate headers/footers, normalize whitespace)
- Add content deduplication (hash-based or MinHash/LSH for near-duplicates)
- Connect cleaned output to tokenizer / training pipeline

**Notes on this approach:**
- Row-group-at-a-time processing keeps RAM usage under ~100MB regardless of file size
- Same code scales from 5 files to 500+ files
- Cleaning config is a simple dict — easy to version and reproduce experiments